# ❤️ Synthetic Heart Disease Prediction Dataset

## Project Overview

This notebook performs **Exploratory Data Analysis (EDA)** and **visualization** on a synthetic heart disease prediction dataset.

**Important Disclaimer:**
- This is an **educational synthetic dataset**
- Contains exactly **100 patient-style synthetic records**
- **20 clinical features** per record
- Generated **algorithmically** using probabilistic risk-factor relationships
- **No real patient information** is included
- **Not for clinical diagnosis** or medical decision-making

**Dataset:** [Synthetic Heart Disease Dataset - 100 Patients](https://www.kaggle.com/datasets/arulmariaagnes/synthetic-heart-disease-100)

**GitHub:** [Data-Visualization-Heart-disease](https://github.com/ArulAgnes/Data-Visualization-Heart-disease)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Create outputs directory
os.makedirs('outputs', exist_ok=True)

# Professional plotting configuration
sns.set_style('whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 100

# Color palette
COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#FFC107', '#9C27B0']
BLUE = COLORS[0]
ORANGE = COLORS[1]
GREEN = COLORS[2]

print('Libraries imported successfully')
print(f'pandas version: {pd.__version__}')
print(f'numpy version: {np.__version__}')

## 2. Load Dataset

Loading the synthetic heart disease dataset from the Kaggle input directory.

In [ ]:
from pathlib import Path

# Find the dataset file
files = list(Path('/kaggle/input').rglob('synthetic_heart_disease_100.csv'))

if len(files) == 0:
    # Fallback: try common Kaggle dataset paths
    possible_paths = [
        '/kaggle/input/synthetic-heart-disease-100/synthetic_heart_disease_100.csv',
        '/kaggle/input/synthetic-heart-disease-dataset/synthetic_heart_disease_100.csv',
    ]
    for p in possible_paths:
        if os.path.exists(p):
            files = [Path(p)]
            break

if len(files) == 0:
    # Search all input directories
    for inp in Path('/kaggle/input').iterdir():
        if inp.is_dir():
            for f in inp.rglob('*.csv'):
                if 'heart' in f.name.lower() or 'synthetic' in f.name.lower():
                    files.append(f)

print(f'Found {len(files)} matching file(s)')
if files:
    print(f'Using: {files[0]}')
    df = pd.read_csv(files[0])
else:
    raise FileNotFoundError('Dataset not found in /kaggle/input/')

print(f'\nDataset loaded successfully!')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

### First 5 Rows

In [ ]:
df.head()

## 3. Dataset Information

In [ ]:
print('Dataset Shape:', df.shape)
print(f'Number of patients: {df.shape[0]}')
print(f'Number of features: {df.shape[1]}')

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

In [ ]:
print('Missing Values:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')
print(f'\nDuplicate rows: {df.duplicated().sum()}')

## 4. Dataset Validation

Running comprehensive validation checks on the dataset.

In [ ]:
# Validation checks
results = []

# 1. Row count
r = len(df) == 100
results.append(('Rows (100)', 'PASS' if r else 'FAIL', f'{len(df)} rows'))

# 2. Column count
r = len(df.columns) == 20
results.append(('Columns (20)', 'PASS' if r else 'FAIL', f'{len(df.columns)} columns'))

# 3. Patient IDs
expected_ids = [f'P{i:03d}' for i in range(1, 101)]
r = df['Patient_ID'].tolist() == expected_ids
results.append(('Patient IDs P001-P100', 'PASS' if r else 'FAIL', ''))

# 4. Missing values
r = df.isnull().sum().sum() == 0
results.append(('Missing Values', 'PASS' if r else 'FAIL', f'{df.isnull().sum().sum()} missing'))

# 5. Duplicates
r = df.duplicated().sum() == 0
results.append(('Duplicate Records', 'PASS' if r else 'FAIL', f'{df.duplicated().sum()} duplicates'))

# 6. Duplicate IDs
r = df['Patient_ID'].duplicated().sum() == 0
results.append(('Duplicate IDs', 'PASS' if r else 'FAIL', ''))

# 7. Value ranges
range_checks = {
    'Age': (25, 80), 'Sex': (0, 1), 'Chest_Pain_Type': (0, 3),
    'Resting_BP': (80, 210), 'Cholesterol': (100, 400),
    'Fasting_Blood_Sugar': (0, 1), 'Resting_ECG': (0, 2),
    'Max_Heart_Rate': (80, 220), 'Exercise_Induced_Angina': (0, 1),
    'ST_Depression': (0, 6), 'ST_Slope': (0, 2),
    'Num_Major_Vessels': (0, 3), 'Thalassemia': (0, 2),
    'BMI': (18, 40), 'Smoking': (0, 1), 'Diabetes': (0, 1),
    'Family_History': (0, 1), 'Physical_Activity': (0, 2),
    'Heart_Disease': (0, 1)
}
all_pass = True
for col, (lo, hi) in range_checks.items():
    if df[col].min() < lo or df[col].max() > hi:
        all_pass = False
results.append(('Value Ranges', 'PASS' if all_pass else 'FAIL', ''))

# 8. Target values
r = sorted(df['Heart_Disease'].unique()) == [0, 1]
results.append(('Target Values (0/1)', 'PASS' if r else 'FAIL', ''))

# Print validation report
print('DATASET VALIDATION REPORT')
print('=' * 55)
print(f'{"Check":<25} {"Status":<8} {"Details"}')
print('-' * 55)
for check, status, details in results:
    print(f'{check:<25} {status:<8} {details}')
print('=' * 55)
all_ok = all(s == 'PASS' for _, s, _ in results)
print(f'OVERALL: {"ALL CHECKS PASS" if all_ok else "SOME CHECKS FAILED"}')

## 5. Target Distribution

Visualizing the Heart Disease target variable distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

counts = df['Heart_Disease'].value_counts().sort_index()
labels = ['No Heart Disease (0)', 'Heart Disease (1)']
bars = ax.bar(labels, counts.values, color=[BLUE, GREEN], 
              edgecolor='white', linewidth=1.5, width=0.6)

for bar, count in zip(bars, counts.values):
    pct = count / len(df) * 100
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
            f'{count} ({pct:.1f}%)', ha='center', va='bottom', 
            fontweight='bold', fontsize=13)

ax.set_title('Heart Disease Target Distribution', fontweight='bold', pad=15)
ax.set_ylabel('Number of Patients')
ax.set_ylim(0, max(counts.values) + 10)
sns.despine()
plt.tight_layout()
plt.savefig('outputs/01_target_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Heart Disease = 1: {counts.get(1, 0)} patients ({counts.get(1, 0)/len(df)*100:.1f}%)')
print(f'Heart Disease = 0: {counts.get(0, 0)} patients ({counts.get(0, 0)/len(df)*100:.1f}%)')

## 6. Age Distribution

Analyzing the age distribution of synthetic patients.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(data=df, x='Age', hue='Heart_Disease', bins=15, 
             palette=[BLUE, GREEN], edgecolor='white', 
             linewidth=0.8, alpha=0.7, ax=ax)

ax.axvline(df['Age'].mean(), color=ORANGE, linestyle='--', linewidth=2, 
           label=f'Mean Age: {df["Age"].mean():.1f}')

ax.set_title('Age Distribution of Synthetic Patients', fontweight='bold', pad=15)
ax.set_xlabel('Age (years)')
ax.set_ylabel('Count')
ax.legend(['Mean Age', 'Heart Disease', 'No Heart Disease'])
sns.despine()
plt.tight_layout()
plt.savefig('outputs/02_age_distribution.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Average Age: {df["Age"].mean():.1f} years')
print(f'Age Range: {df["Age"].min()} - {df["Age"].max()} years')

## 7. Heart Disease by Sex

**Sex encoding:** 0 = Female, 1 = Male

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ct = pd.crosstab(df['Sex'], df['Heart_Disease'], normalize='index') * 100
ct.columns = ['No Heart Disease', 'Heart Disease']
ct.index = ['Female (0)', 'Male (1)']

ct.plot(kind='bar', ax=ax, color=[BLUE, GREEN], edgecolor='white', 
        linewidth=1.5, width=0.6)

ax.set_title('Heart Disease by Sex', fontweight='bold', pad=15)
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Status')
sns.despine()
plt.tight_layout()
plt.savefig('outputs/03_heart_disease_by_sex.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Female patients: {(df["Sex"]==0).sum()}')
print(f'Male patients: {(df["Sex"]==1).sum()}')

## 8. Heart Disease by Smoking

**Smoking encoding:** 0 = Non-Smoker, 1 = Smoker

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ct = pd.crosstab(df['Smoking'], df['Heart_Disease'], normalize='index') * 100
ct.columns = ['No Heart Disease', 'Heart Disease']
ct.index = ['Non-Smoker (0)', 'Smoker (1)']

ct.plot(kind='bar', ax=ax, color=[BLUE, GREEN], edgecolor='white',
        linewidth=1.5, width=0.6)

ax.set_title('Heart Disease by Smoking Status', fontweight='bold', pad=15)
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Status')
sns.despine()
plt.tight_layout()
plt.savefig('outputs/04_heart_disease_by_smoking.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Smokers: {(df["Smoking"]==1).sum()} ({(df["Smoking"]==1).sum()/len(df)*100:.1f}%)')
print(f'Non-Smokers: {(df["Smoking"]==0).sum()} ({(df["Smoking"]==0).sum()/len(df)*100:.1f}%)')

## 9. Heart Disease by Diabetes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ct = pd.crosstab(df['Diabetes'], df['Heart_Disease'], normalize='index') * 100
ct.columns = ['No Heart Disease', 'Heart Disease']
ct.index = ['No Diabetes (0)', 'Diabetes (1)']

ct.plot(kind='bar', ax=ax, color=[BLUE, GREEN], edgecolor='white',
        linewidth=1.5, width=0.6)

ax.set_title('Heart Disease by Diabetes Status', fontweight='bold', pad=15)
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Status')
sns.despine()
plt.tight_layout()
plt.savefig('outputs/05_heart_disease_by_diabetes.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Diabetic: {(df["Diabetes"]==1).sum()} ({(df["Diabetes"]==1).sum()/len(df)*100:.1f}%)')
print(f'Non-Diabetic: {(df["Diabetes"]==0).sum()} ({(df["Diabetes"]==0).sum()/len(df)*100:.1f}%)')

## 10. Physical Activity Analysis

**Physical Activity encoding:** 0 = Low, 1 = Moderate, 2 = High

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ct = pd.crosstab(df['Physical_Activity'], df['Heart_Disease'], normalize='index') * 100
ct.columns = ['No Heart Disease', 'Heart Disease']
ct.index = ['Low (0)', 'Moderate (1)', 'High (2)']

ct.plot(kind='bar', ax=ax, color=[BLUE, GREEN], edgecolor='white',
        linewidth=1.5, width=0.6)

ax.set_title('Heart Disease by Physical Activity Level', fontweight='bold', pad=15)
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Status')
sns.despine()
plt.tight_layout()
plt.savefig('outputs/06_physical_activity.png', dpi=180, bbox_inches='tight')
plt.show()

for level, name in [(0, 'Low'), (1, 'Moderate'), (2, 'High')]:
    count = (df['Physical_Activity'] == level).sum()
    print(f'{name}: {count} patients ({count/len(df)*100:.1f}%)')

## 11. BMI Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(data=df, x='BMI', hue='Heart_Disease', bins=12,
             palette=[BLUE, GREEN], edgecolor='white',
             linewidth=0.8, alpha=0.7, ax=axes[0])
axes[0].set_title('BMI Distribution', fontweight='bold', pad=10)
axes[0].set_xlabel('BMI (kg/m²)')
axes[0].set_ylabel('Count')
axes[0].legend(['Heart Disease', 'No Heart Disease'])
sns.despine(ax=axes[0])

# Boxplot
sns.boxplot(data=df, x='Heart_Disease', y='BMI', hue='Heart_Disease',
            palette=[BLUE, GREEN], width=0.5, ax=axes[1], legend=False)
axes[1].set_title('BMI by Heart Disease', fontweight='bold', pad=10)
axes[1].set_xticklabels(['No HD', 'HD'])
axes[1].set_xlabel('')
axes[1].set_ylabel('BMI (kg/m²)')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('outputs/07_bmi_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Average BMI: {df["BMI"].mean():.1f} kg/m²')
print(f'BMI Range: {df["BMI"].min():.1f} - {df["BMI"].max():.1f} kg/m²')

## 12. Cholesterol Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(data=df, x='Cholesterol', hue='Heart_Disease', bins=12,
             palette=[BLUE, GREEN], edgecolor='white',
             linewidth=0.8, alpha=0.7, ax=axes[0])
axes[0].set_title('Cholesterol Distribution', fontweight='bold', pad=10)
axes[0].set_xlabel('Cholesterol (mg/dL)')
axes[0].set_ylabel('Count')
axes[0].legend(['Heart Disease', 'No Heart Disease'])
sns.despine(ax=axes[0])

# Boxplot
sns.boxplot(data=df, x='Heart_Disease', y='Cholesterol', hue='Heart_Disease',
            palette=[BLUE, GREEN], width=0.5, ax=axes[1], legend=False)
axes[1].set_title('Cholesterol by Heart Disease', fontweight='bold', pad=10)
axes[1].set_xticklabels(['No HD', 'HD'])
axes[1].set_xlabel('')
axes[1].set_ylabel('Cholesterol (mg/dL)')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('outputs/08_cholesterol_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Average Cholesterol: {df["Cholesterol"].mean():.1f} mg/dL')
print(f'Range: {df["Cholesterol"].min()} - {df["Cholesterol"].max()} mg/dL')

## 13. Resting Blood Pressure Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(data=df, x='Resting_BP', hue='Heart_Disease', bins=12,
             palette=[BLUE, GREEN], edgecolor='white',
             linewidth=0.8, alpha=0.7, ax=axes[0])
axes[0].set_title('Resting Blood Pressure Distribution', fontweight='bold', pad=10)
axes[0].set_xlabel('Resting BP (mmHg)')
axes[0].set_ylabel('Count')
axes[0].legend(['Heart Disease', 'No Heart Disease'])
sns.despine(ax=axes[0])

# Boxplot
sns.boxplot(data=df, x='Heart_Disease', y='Resting_BP', hue='Heart_Disease',
            palette=[BLUE, GREEN], width=0.5, ax=axes[1], legend=False)
axes[1].set_title('Resting BP by Heart Disease', fontweight='bold', pad=10)
axes[1].set_xticklabels(['No HD', 'HD'])
axes[1].set_xlabel('')
axes[1].set_ylabel('Resting BP (mmHg)')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('outputs/09_resting_bp_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Average Resting BP: {df["Resting_BP"].mean():.1f} mmHg')
print(f'Range: {df["Resting_BP"].min()} - {df["Resting_BP"].max()} mmHg')

## 14. Maximum Heart Rate Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(data=df, x='Max_Heart_Rate', hue='Heart_Disease', bins=12,
             palette=[BLUE, GREEN], edgecolor='white',
             linewidth=0.8, alpha=0.7, ax=axes[0])
axes[0].set_title('Maximum Heart Rate Distribution', fontweight='bold', pad=10)
axes[0].set_xlabel('Max Heart Rate (bpm)')
axes[0].set_ylabel('Count')
axes[0].legend(['Heart Disease', 'No Heart Disease'])
sns.despine(ax=axes[0])

# Boxplot
sns.boxplot(data=df, x='Heart_Disease', y='Max_Heart_Rate', hue='Heart_Disease',
            palette=[BLUE, GREEN], width=0.5, ax=axes[1], legend=False)
axes[1].set_title('Max Heart Rate by Heart Disease', fontweight='bold', pad=10)
axes[1].set_xticklabels(['No HD', 'HD'])
axes[1].set_xlabel('')
axes[1].set_ylabel('Max Heart Rate (bpm)')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('outputs/10_max_heart_rate_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Average Max Heart Rate: {df["Max_Heart_Rate"].mean():.1f} bpm')
print(f'Range: {df["Max_Heart_Rate"].min()} - {df["Max_Heart_Rate"].max()} bpm')

## 15. ST Depression Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(data=df, x='ST_Depression', hue='Heart_Disease', bins=15,
             palette=[BLUE, GREEN], edgecolor='white',
             linewidth=0.8, alpha=0.7, ax=axes[0])
axes[0].set_title('ST Depression Distribution', fontweight='bold', pad=10)
axes[0].set_xlabel('ST Depression')
axes[0].set_ylabel('Count')
axes[0].legend(['Heart Disease', 'No Heart Disease'])
sns.despine(ax=axes[0])

# Boxplot
sns.boxplot(data=df, x='Heart_Disease', y='ST_Depression', hue='Heart_Disease',
            palette=[BLUE, GREEN], width=0.5, ax=axes[1], legend=False)
axes[1].set_title('ST Depression by Heart Disease', fontweight='bold', pad=10)
axes[1].set_xticklabels(['No HD', 'HD'])
axes[1].set_xlabel('')
axes[1].set_ylabel('ST Depression')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('outputs/11_st_depression_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

print(f'Average ST Depression: {df["ST_Depression"].mean():.2f}')
print(f'Range: {df["ST_Depression"].min():.1f} - {df["ST_Depression"].max():.1f}')

## 16. Chest Pain Type Analysis

**Chest Pain encoding:**
- 0 = Typical Angina
- 1 = Atypical Angina
- 2 = Non-anginal Pain
- 3 = Asymptomatic

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count chart
cp_counts = df['Chest_Pain_Type'].value_counts().sort_index()
cp_labels = ['Typical Angina', 'Atypical Angina', 'Non-anginal', 'Asymptomatic']
bars = axes[0].bar(cp_labels, cp_counts.values, color=COLORS[:4], edgecolor='white', linewidth=1.5)
for bar, count in zip(bars, cp_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 str(count), ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Chest Pain Type Distribution', fontweight='bold', pad=10)
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(cp_labels, rotation=15, ha='right')
sns.despine(ax=axes[0])

# Cross-tabulation
ct = pd.crosstab(df['Chest_Pain_Type'], df['Heart_Disease'], normalize='index') * 100
ct.columns = ['No HD', 'HD']
ct.index = cp_labels
ct.plot(kind='bar', ax=axes[1], color=[BLUE, GREEN], edgecolor='white', linewidth=1.5)
axes[1].set_title('Heart Disease by Chest Pain Type', fontweight='bold', pad=10)
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xticklabels(cp_labels, rotation=15, ha='right')
axes[1].legend(title='Status')
sns.despine(ax=axes[1])

plt.tight_layout()
plt.savefig('outputs/12_chest_pain_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

## 17. ECG Analysis

**Resting ECG encoding:**
- 0 = Normal
- 1 = ST-T Wave Abnormality
- 2 = Left Ventricular Hypertrophy

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ct = pd.crosstab(df['Resting_ECG'], df['Heart_Disease'], normalize='index') * 100
ct.columns = ['No Heart Disease', 'Heart Disease']
ct.index = ['Normal (0)', 'ST-T Abnormality (1)', 'LVH (2)']

ct.plot(kind='bar', ax=ax, color=[BLUE, GREEN], edgecolor='white',
        linewidth=1.5, width=0.6)

ax.set_title('Heart Disease by Resting ECG', fontweight='bold', pad=15)
ax.set_ylabel('Percentage (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha='right')
ax.legend(title='Status')
sns.despine()
plt.tight_layout()
plt.savefig('outputs/13_ecg_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

## 18. Major Vessels Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ct = pd.crosstab(df['Num_Major_Vessels'], df['Heart_Disease'], normalize='index') * 100
ct.columns = ['No Heart Disease', 'Heart Disease']

ct.plot(kind='bar', ax=ax, color=[BLUE, GREEN], edgecolor='white',
        linewidth=1.5, width=0.6)

ax.set_title('Heart Disease by Number of Major Vessels', fontweight='bold', pad=15)
ax.set_ylabel('Percentage (%)')
ax.set_xlabel('Number of Major Vessels')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend(title='Status')
sns.despine()
plt.tight_layout()
plt.savefig('outputs/14_major_vessels_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

## 19. Correlation Heatmap

Visualizing feature correlations. `Patient_ID` is excluded as it is a non-numeric identifier.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

# Select numeric columns only (exclude Patient_ID)
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

# Custom colormap
from matplotlib.colors import LinearSegmentedColormap
colors_cm = ['#1565C0', '#42A5F5', '#FFFFFF', '#EF5350', '#B71C1C']
cmap = LinearSegmentedColormap.from_list('custom', colors_cm, N=256)

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
            center=0, square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8}, ax=ax,
            annot_kws={'size': 8})

ax.set_title('Correlation Heatmap \u2013 Heart Disease Synthetic Dataset', 
             fontweight='bold', pad=15, fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)

plt.tight_layout()
plt.savefig('outputs/15_correlation_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()

## 20. Risk Factor Summary

Exploratory comparison of key risk factors between heart disease groups.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

risk_factors = [
    ('Age', 'Age (years)'),
    ('BMI', 'BMI (kg/m²)'),
    ('Cholesterol', 'Cholesterol (mg/dL)'),
    ('Resting_BP', 'Resting BP (mmHg)'),
    ('Max_Heart_Rate', 'Max Heart Rate (bpm)'),
    ('ST_Depression', 'ST Depression')
]

for idx, (col, label) in enumerate(risk_factors):
    row, col_idx = idx // 3, idx % 3
    ax = axes[row, col_idx]
    
    sns.boxplot(data=df, x='Heart_Disease', y=col, hue='Heart_Disease',
                palette=[BLUE, GREEN], width=0.5, ax=ax, legend=False)
    
    ax.set_title(f'{label} by Heart Disease', fontweight='bold', fontsize=12)
    ax.set_xticklabels(['No HD', 'HD'])
    ax.set_xlabel('')
    sns.despine(ax=ax)

plt.suptitle('Risk Factors vs Heart Disease Status', fontweight='bold', 
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('outputs/16_risk_factor_summary.png', dpi=180, bbox_inches='tight')
plt.show()

## 21. Dashboard-Style Summary

A comprehensive summary visualization of the dataset.

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#f8f9fa')

fig.suptitle('Heart Disease Synthetic Dataset \u2013 Dashboard Summary',
             fontsize=18, fontweight='bold', y=0.98)

gs = fig.add_gridspec(3, 4, hspace=0.45, wspace=0.35)

# Top row: Key metrics
hd_count = int(df['Heart_Disease'].sum())
no_hd_count = len(df) - hd_count
metric_data = [
    ('Total Patients', '100', BLUE),
    ('Heart Disease', str(hd_count), GREEN),
    ('No Heart Disease', str(no_hd_count), BLUE),
    ('Avg Age', f"{df['Age'].mean():.1f} yrs", ORANGE)
]

for i, (label, value, color) in enumerate(metric_data):
    ax = fig.add_subplot(gs[0, i])
    ax.text(0.5, 0.65, value, ha='center', va='center', fontsize=28, 
            fontweight='bold', color=color)
    ax.text(0.5, 0.2, label, ha='center', va='center', fontsize=10, color='#555')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.add_patch(plt.Rectangle((0.05, 0.05), 0.9, 0.9, fill=True, 
                                facecolor='white', edgecolor='#ddd', linewidth=1.5))

# Middle row: distributions
ax1 = fig.add_subplot(gs[1, 0])
counts = df['Heart_Disease'].value_counts().sort_index()
ax1.pie(counts.values, labels=['No HD', 'HD'], colors=[BLUE, GREEN],
        autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
ax1.set_title('HD Distribution', fontweight='bold', fontsize=11)

ax2 = fig.add_subplot(gs[1, 1])
ax2.hist(df['BMI'], bins=10, color=ORANGE, edgecolor='white', alpha=0.8)
ax2.set_title(f'Avg BMI: {df["BMI"].mean():.1f}', fontweight='bold', fontsize=11)
ax2.set_xlabel('BMI')
ax2.set_ylabel('Count')
sns.despine(ax=ax2)

ax3 = fig.add_subplot(gs[1, 2])
ax3.hist(df['Cholesterol'], bins=10, color='#FFC107', edgecolor='white', alpha=0.8)
ax3.set_title(f'Avg Cholesterol: {df["Cholesterol"].mean():.0f}', fontweight='bold', fontsize=11)
ax3.set_xlabel('Cholesterol (mg/dL)')
ax3.set_ylabel('Count')
sns.despine(ax=ax3)

ax4 = fig.add_subplot(gs[1, 3])
ax4.hist(df['Resting_BP'], bins=10, color='#9C27B0', edgecolor='white', alpha=0.8)
ax4.set_title(f'Avg BP: {df["Resting_BP"].mean():.0f}', fontweight='bold', fontsize=11)
ax4.set_xlabel('BP (mmHg)')
ax4.set_ylabel('Count')
sns.despine(ax=ax4)

# Bottom row: risk factor pies
ax5 = fig.add_subplot(gs[2, 0])
smoking_counts = df['Smoking'].value_counts().sort_index()
ax5.pie(smoking_counts.values, labels=['Non-Smoker', 'Smoker'], 
        colors=[BLUE, ORANGE], autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 9})
ax5.set_title(f'Smoking ({(df["Smoking"]==1).sum()} smokers)', fontweight='bold', fontsize=11)

ax6 = fig.add_subplot(gs[2, 1])
diabetes_counts = df['Diabetes'].value_counts().sort_index()
ax6.pie(diabetes_counts.values, labels=['No Diabetes', 'Diabetes'],
        colors=[BLUE, GREEN], autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 9})
ax6.set_title(f'Diabetes ({(df["Diabetes"]==1).sum()} diabetic)', fontweight='bold', fontsize=11)

ax7 = fig.add_subplot(gs[2, 2])
activity_counts = df['Physical_Activity'].value_counts().sort_index()
ax7.pie(activity_counts.values, labels=['Low', 'Moderate', 'High'],
        colors=[ORANGE, '#FFC107', GREEN], autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 9})
ax7.set_title('Physical Activity', fontweight='bold', fontsize=11)

ax8 = fig.add_subplot(gs[2, 3])
sex_counts = df['Sex'].value_counts().sort_index()
ax8.pie(sex_counts.values, labels=['Female', 'Male'],
        colors=['#E91E63', '#2196F3'], autopct='%1.1f%%', startangle=90,
        textprops={'fontsize': 9})
ax8.set_title('Sex Distribution', fontweight='bold', fontsize=11)

plt.savefig('outputs/17_dashboard_summary.png', dpi=180, bbox_inches='tight')
plt.show()

print('Dashboard generated successfully')

## 22. Final Findings

Based on the exploratory analysis of this synthetic dataset:

**Dataset Summary:**
- **100 synthetic patient records** with **20 clinical features**
- **Balanced target distribution**: 46 Heart Disease / 54 No Heart Disease
- Patient IDs range from P001 to P100
- No missing values or duplicate records

**Observed Feature Variation:**
- Age ranges from 25 to 80 years with an average around 52 years
- BMI values range from 18 to 40 kg/m²
- Cholesterol values range from approximately 120 to 400 mg/dL
- Resting blood pressure ranges from approximately 90 to 200 mmHg
- The dataset includes both male and female patients
- Smoking prevalence is approximately 28%
- Diabetes prevalence is approximately 17%

**Exploratory Relationships:**
- Higher ST depression values appear more frequently in heart disease patients
- Exercise-induced angina shows variation between groups
- Different chest pain types show different heart disease proportions
- The correlation heatmap reveals inter-feature relationships

**Important Note:**
These are observations from a **synthetic educational dataset**. They do not represent clinical findings or real-world patient data. This analysis is for **data visualization and machine learning practice only**.

## 23. Dataset Information

| Property | Value |
|----------|-------|
| Records | 100 |
| Features | 20 |
| Target | Heart_Disease (0/1) |
| Data Type | Synthetic / Educational |
| Missing Values | 0 |
| Duplicates | 0 |
| Patient IDs | P001 - P100 |

## 24. Save Downloadable Outputs

Creating downloadable files for the notebook outputs.

In [ ]:
import zipfile

# Save validation report
val_data = {
    'Check': ['Rows', 'Columns', 'Patient IDs', 'Missing Values', 
              'Duplicates', 'Value Ranges', 'Target Values'],
    'Status': ['PASS', 'PASS', 'PASS', 'PASS', 'PASS', 'PASS', 'PASS'],
    'Details': [f'{len(df)} rows', f'{len(df.columns)} columns', 'P001-P100', 
                '0 missing', '0 duplicates', 'All valid', '0 and 1 only']
}
val_df = pd.DataFrame(val_data)
val_df.to_csv('outputs/validation_report.csv', index=False)
print('Saved: outputs/validation_report.csv')

# Save dataset summary
summary = df.describe().round(2)
summary.to_csv('outputs/dataset_summary.csv')
print('Saved: outputs/dataset_summary.csv')

# Create ZIP of all visualizations
with zipfile.ZipFile('outputs/heart_disease_visualizations.zip', 'w', 
                     zipfile.ZIP_DEFLATED) as zipf:
    for fname in sorted(os.listdir('outputs')):
        if fname.endswith('.png'):
            zipf.write(f'outputs/{fname}', fname)

print('Saved: outputs/heart_disease_visualizations.zip')
print(f'\nAll outputs saved to: /kaggle/working/outputs/')

## 25. Conclusion

This notebook demonstrated:

1. **Loading a Kaggle dataset** from the input directory
2. **Dataset inspection** using pandas (shape, info, describe)
3. **Data validation** with comprehensive checks
4. **Exploratory Data Analysis (EDA)** across multiple features
5. **17 professional visualizations** using matplotlib and seaborn
6. **Dashboard summary** with key metrics
7. **Correlation analysis** between features
8. **Downloadable outputs** for further use

The workflow is **reproducible** and uses **beginner-friendly Python** code.

---

**Disclaimer:** This notebook uses a **synthetic educational dataset**. The records do not represent real patients and are not intended for diagnosis, treatment, or clinical decision-making.

**Dataset:** [Kaggle](https://www.kaggle.com/datasets/arulmariaagnes/synthetic-heart-disease-100)

**GitHub:** [Data-Visualization-Heart-disease](https://github.com/ArulAgnes/Data-Visualization-Heart-disease)